1. Which linear regression training algorithm can you use if you have a training set
with millions of features?

- SGD или MiniBatchGD комбинирано с Lasso regression

2. Suppose the features in your training set have very different scales. Which algo‐
rithms might suffer from this, and how? What can you do about it?

- Всички GD алгоритми - при тях ако features не са скалирани пътят към глобалния минимум става по дълъг и може и да не успее да го достигне.
За да се избегне може да се използва StandardScaler за да е сигурно че са в един диапазон

3. Can gradient descent get stuck in a local minimum when training a logistic
regression model?

- няма как защото той използва за loss изпъкнала фунция и няма локален минимум

4. Do all gradient descent algorithms lead to the same model, provided you let them
run long enough?

- при достатъчно много итерации и изпъкнала loss функция би трябвало да достигнат почти еднакви стойности на параметрите

5. Suppose you use batch gradient descent and you plot the validation error at every
epoch. If you notice that the validation error consistently goes up, what is likely
going on? How can you fix this?

- това означава че моделът започва да overfit данните 
- ще подам хиперпараметъра early_stopping=True за да мога да предотвратя overfit и да взема модела с най добре генерализиращи параметри

6. Is it a good idea to stop mini-batch gradient descent immediately when the
validation error goes up?

- не е добра идея защото има опастност модела да не е видял всичките данни 
- ако ще се използва early_stopping=True бих подал по голяма стойност за n_iter_no_change за да съм сигурен че съм достигнал истинския минимун на кост функцията на валидиращия сет 

7. Which gradient descent algorithm (among those we discussed) will reach the
vicinity of the optimal solution the fastest? Which will actually converge? How
can you make the others converge as well?

- най бързо ще намери оптималните параметри Batch GD и ще се converge-не най добре 
- за да се получи максимално подобен ефект и при SGD и MiniBatch GD трябва да има добре подбран learning schedule за да може алгоритъма колкото повече напредва в итерациите да прави по малки стъпки за да може по бързо да се converge-не

8. Suppose you are using polynomial regression. You plot the learning curves and
you notice that there is a large gap between the training error and the validation
error. What is happening? What are three ways to solve this?

- това означава че модела прави overfitting -> има high variance
- 1. полагане на регуляризация върху модела
- 2. избиране на по прост модел
- 3. обработване на данние като скалиране или премахване на шума или оголемяване на сета

9. Suppose you are using ridge regression and you notice that the training error
and the validation error are almost equal and fairly high. Would you say that
the model suffers from high bias or high variance? Should you increase the
regularization hyperparameter α or reduce it?

- в тази означава че има high bias -> underfit -> моделът е прекалено прост и не може да научи главните зависимости в данните
- hyperparameter α трябва да се намали за да може да се даде повече свобода на модела да при намиране на подходящи параметри

10. Why would you want to use:
a. Ridge regression instead of plain linear regression (i.e., without any
regularization)?
b. Lasso instead of ridge regression?
c. Elastic net instead of lasso regression?

- а. -> за да предотвратя overfitting като положа регуляризация върху модела
- b. -> за да мога едновременно да направя регуляризация и селекция на най важните features 
- c. -> ако Lasso е прекалено агресивен и ми премахва важни features но все пак искам да ми направи селекция и едновременно с това искам да мога да редуцирам стойността на параметрите с ridge

11. Suppose you want to classify pictures as outdoor/indoor and daytime/nighttime.
Should you implement two logistic regression classifiers or one softmax regres‐
sion classifier?

- трабва да се имплементират два класификатора защото softmax regression classifier е multiclass и трябва да се използва само с взаимно изключващи се класове, като не може да бъде multioutput защото предсказва само един клас за даден инстанс 

12. Implement batch gradient descent with early stopping for softmax regression
without using Scikit-Learn, only NumPy. Use it on a classification task such as
the iris dataset.

In [121]:
from sklearn.datasets import load_iris
import numpy as np
np.random.seed(13)

iris = load_iris(as_frame=True)

X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = iris.target.values
target_labels = np.sort(iris.target_names)

In [122]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [123]:
def one_hot(y):
    y_one_hot = np.zeros((y.shape[0], len(target_labels)))
    for i in range(len(target_labels)):
        y_one_hot[y == i, i] = 1

    return y_one_hot

n_samples = X.shape[0]
shuffled_inxs = np.random.permutation(n_samples)
test_ratio = 0.2
test_len = int(n_samples * test_ratio)

# X standard scale and add dummy feature
X_scale = (X - X.mean(axis=0)) / X.std(axis=0)
X_b = np.empty((X.shape[0], X.shape[1] + 1))
X_b[:, 1:] = X_scale
X_b[:, 0] = 1

# split train test sets
X_b = X_b[shuffled_inxs]
y = y[shuffled_inxs]

X_train, y_train = X_b[test_len:], y[test_len:]
X_test, y_test = X_b[:test_len], y[:test_len]

y_train = one_hot(y_train)

# split train validation sets
val_set_ratio = 0.1
val_set_len = int(len(X_train) * val_set_ratio)

X_val, y_val = X_train[:val_set_len], y_train[:val_set_len]
X_train, y_train = X_train[val_set_len:], y_train[val_set_len:]

In [139]:
def log_func(reg_score):
    classes_exp = np.exp(reg_score)
    return classes_exp / classes_exp.sum(axis=1, keepdims=True)

def cost_func(y, p):
    return -(y * np.log(p)).sum(axis=1).mean()

n_epochs = 10000
tol = 0.01
n_iter_no_change = 5
eta = 0.1
m = X_train.shape[0]

best_score = np.inf
best_theta = None
counter = 0
theta = np.random.randn(len(target_labels), X_train.shape[1])

for epoch in range(n_epochs):
    p = log_func(X_train @ theta.T)
    gradiends = ((p - y_train).T @ X_train) / m
    theta -= eta * gradiends

    score = cost_func(y_val, log_func(X_val @ theta.T))
    if np.abs(score - best_score) > tol:
        best_score = score
        best_theta = theta
        counter = 0

    else:
        counter += 1
        if counter == n_iter_no_change:
            break

print('n_iter:', epoch + 1)

n_iter: 59


In [140]:
def accuracy(real_data, predicted_data):
    n_correct = (predicted_data == real_data).sum()
    return n_correct / len(real_data)

pred_proba = log_func(X_test @ best_theta.T)
y_test_pred = pred_proba.argmax(axis=1)

accuracy(y_test, y_test_pred)

np.float64(0.9666666666666667)